# Notebook 12 — Sauvegarde des modèles pour le pipeline

## Objectif

Entraîner et sauvegarder les 6 modèles utilisés par le pipeline :

| Modèle | Système | Fichier |
|--------|---------|---------|
| LOF | Train Ticket | `models/lof_tt.pkl` |
| LOF | Online Boutique | `models/lof_ob.pkl` |
| TF-IDF | Train Ticket | `models/tfidf_tt.pkl` |
| TF-IDF | Online Boutique | `models/tfidf_ob.pkl` |
| IF par service | Train Ticket | `models/if

In [1]:
import pickle
import csv
import re
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import timedelta
from collections import Counter
from sklearn.neighbors import LocalOutlierFactor
from sklearn.ensemble import IsolationForest
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

PROJET    = Path('/home/eunice/Bureau/Train_ticket/Intelligent_observability')
NORMAL    = PROJET / 'data/normal'
MODELS_DIR = PROJET / 'models'
MODELS_DIR.mkdir(exist_ok=True)

METRIQUES = [
    'CpuUsageRate(%)',
    'MemoryUsageRate(%)',
    'PodServerLatencyP99(s)',
    'NetworkReceiveBytes',
    'NetworkTransmitBytes',
]
FEATURES_SPAN = ['duration_ms']

# ─── FONCTIONS DE CHARGEMENT ───
def charger_metriques(date, source):
    metric_dir = source / date / 'metric'
    if not metric_dir.exists():
        return pd.DataFrame()
    dfs = []
    for f in sorted(metric_dir.glob('*_metric.csv')):
        service = f.stem.rsplit('-', 2)[0]
        df = pd.read_csv(f)
        df['service'] = service
        df['datetime'] = pd.to_datetime(df['TimeStamp'], unit='s', utc=True)
        for col in df.columns:
            if col not in ['Time', 'PodName', 'service', 'datetime']:
                df[col] = pd.to_numeric(
                    df[col].replace('NaN', float('nan')), errors='coerce')
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

def charger_logs(date, source, fenetre):
    chemin = source / date / 'log' / f'{fenetre}_log.csv'
    if not chemin.exists():
        return pd.DataFrame()
    colonnes = ['Timestamp','TimeUnixNano','Node','PodName',
                'Container','TraceID','SpanID','Log']
    rows = []
    with open(chemin, 'r', encoding='utf-8', errors='replace') as f:
        reader = csv.reader(f)
        next(reader)
        for row in reader:
            if len(row) >= 8:
                rows.append(row[:8])
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows, columns=colonnes)
    df['service'] = df['PodName'].apply(lambda x: str(x).rsplit('-', 2)[0])
    return df

def charger_traces(date, source, fenetre):
    chemin = source / date / 'trace' / f'{fenetre}_trace.csv'
    if not chemin.exists():
        return pd.DataFrame()
    df = pd.read_csv(chemin, on_bad_lines='skip')
    df['duration_ms'] = pd.to_numeric(df['Duration'], errors='coerce') / 1e6
    df['service'] = df['PodName'].apply(lambda x: str(x).rsplit('-', 2)[0])
    return df

def extraire_template(log_str):
    log_str = str(log_str)
    match = re.search(r'"log"\s*:\s*"([^"]+)"', log_str)
    if match: log_str = match.group(1)
    log_str = re.sub(r'[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}', '<UUID>', log_str)
    log_str = re.sub(r'[0-9a-f]{16,}', '<HEX>', log_str)
    log_str = re.sub(r'\b\d+\.?\d*\b', '<NUM>', log_str)
    crochets = re.findall(r'\[([^\]]+)\]', log_str)
    if crochets: return ' | '.join(crochets[:3])
    return ' '.join(log_str.split()[:6])

print("✓ Configuration OK")

✓ Configuration OK


## 1. Modèles Train Ticket

In [2]:
# ═══════════════════════════════════════════
# LOF Train Ticket
# ═══════════════════════════════════════════
print("=== Entraînement LOF Train Ticket ===")

DATES_TT = ['2023-01-29', '2023-01-30']
df_normal_tt = pd.concat([charger_metriques(d, NORMAL) for d in DATES_TT],
                          ignore_index=True)
print(f"  Données : {len(df_normal_tt):,} lignes, {df_normal_tt['service'].nunique()} services")

modeles_lof_tt = {}
scalers_lof_tt = {}

for service in df_normal_tt['service'].unique():
    df_svc = df_normal_tt[df_normal_tt['service'] == service][METRIQUES].dropna()
    if len(df_svc) < 30:
        continue
    scaler = StandardScaler()
    X = scaler.fit_transform(df_svc)
    lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05, novelty=True)
    lof.fit(X)
    modeles_lof_tt[service] = lof
    scalers_lof_tt[service] = scaler

with open(MODELS_DIR / 'lof_tt.pkl', 'wb') as f:
    pickle.dump({
        'modeles': modeles_lof_tt,
        'scalers': scalers_lof_tt,
    }, f)

print(f"  ✓ {len(modeles_lof_tt)} modèles LOF sauvegardés dans lof_tt.pkl")

=== Entraînement LOF Train Ticket ===
  Données : 76,360 lignes, 46 services
  ✓ 46 modèles LOF sauvegardés dans lof_tt.pkl


In [3]:
# ═══════════════════════════════════════════
# TF-IDF Train Ticket
# ═══════════════════════════════════════════
print("=== Entraînement TF-IDF Train Ticket ===")

textes_normaux_tt = []
for date in DATES_TT:
    log_dir = NORMAL / date / 'log'
    if not log_dir.exists():
        continue
    for f in sorted(log_dir.glob('*.csv')):
        df = charger_logs(date, NORMAL, f.stem.replace('_log', ''))
        if df.empty:
            continue
        df['template'] = df['Log'].apply(extraire_template)
        textes_normaux_tt.append(' '.join(df['template'].tolist()))

tfidf_tt = TfidfVectorizer(max_features=500)
tfidf_tt.fit(textes_normaux_tt)
vecteur_ref_tt = np.asarray(tfidf_tt.transform(textes_normaux_tt).mean(axis=0))

with open(MODELS_DIR / 'tfidf_tt.pkl', 'wb') as f:
    pickle.dump({
        'vectorizer'  : tfidf_tt,
        'vecteur_ref' : vecteur_ref_tt,
    }, f)

print(f"  ✓ TF-IDF sauvegardé dans tfidf_tt.pkl")
print(f"    Vocabulaire : {len(tfidf_tt.vocabulary_)} termes")

=== Entraînement TF-IDF Train Ticket ===
  ✓ TF-IDF sauvegardé dans tfidf_tt.pkl
    Vocabulaire : 355 termes


In [4]:
# ═══════════════════════════════════════════
# IF par service Train Ticket
# ═══════════════════════════════════════════
print("=== Entraînement Isolation Forest par service — Train Ticket ===")

modeles_if_traces_tt = {}
scalers_if_traces_tt = {}

for date in DATES_TT:
    trace_dir = NORMAL / date / 'trace'
    if not trace_dir.exists():
        continue
    for f in sorted(trace_dir.glob('*.csv')):
        df = charger_traces(date, NORMAL, f.stem.replace('_trace', ''))
        if df.empty:
            continue
        for service in df['service'].unique():
            df_svc = df[df['service'] == service][FEATURES_SPAN].dropna()
            if len(df_svc) < 5:
                continue
            if service not in modeles_if_traces_tt:
                scaler = StandardScaler()
                X_svc = scaler.fit_transform(df_svc)
                model = IsolationForest(
                    n_estimators=100, contamination=0.10, random_state=42
                )
                model.fit(X_svc)
                modeles_if_traces_tt[service] = model
                scalers_if_traces_tt[service] = scaler

with open(MODELS_DIR / 'if_traces_tt.pkl', 'wb') as f:
    pickle.dump({
        'modeles': modeles_if_traces_tt,
        'scalers': scalers_if_traces_tt,
    }, f)

print(f"  ✓ {len(modeles_if_traces_tt)} modèles IF sauvegardés dans if_traces_tt.pkl")

=== Entraînement Isolation Forest par service — Train Ticket ===
  ✓ 28 modèles IF sauvegardés dans if_traces_tt.pkl


## 2. Modèles Online Boutique

In [5]:
# ═══════════════════════════════════════════
# Modèles Online Boutique (les 3 en une cellule)
# ═══════════════════════════════════════════
DATES_OB = ['2022-08-22', '2022-08-23']

# ─── LOF Online Boutique ───
print("=== LOF Online Boutique ===")
df_normal_ob = pd.concat([charger_metriques(d, NORMAL) for d in DATES_OB],
                          ignore_index=True)
print(f"  Données : {len(df_normal_ob):,} lignes, {df_normal_ob['service'].nunique()} services")

modeles_lof_ob = {}
scalers_lof_ob = {}
for service in df_normal_ob['service'].unique():
    df_svc = df_normal_ob[df_normal_ob['service'] == service][METRIQUES].dropna()
    if len(df_svc) < 30:
        continue
    scaler = StandardScaler()
    X = scaler.fit_transform(df_svc)
    lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05, novelty=True)
    lof.fit(X)
    modeles_lof_ob[service] = lof
    scalers_lof_ob[service] = scaler

with open(MODELS_DIR / 'lof_ob.pkl', 'wb') as f:
    pickle.dump({'modeles': modeles_lof_ob, 'scalers': scalers_lof_ob}, f)
print(f"  ✓ {len(modeles_lof_ob)} modèles LOF sauvegardés")

# ─── TF-IDF Online Boutique ───
print("\n=== TF-IDF Online Boutique ===")
textes_normaux_ob = []
for date in DATES_OB:
    log_dir = NORMAL / date / 'log'
    if not log_dir.exists():
        continue
    for f in sorted(log_dir.glob('*.csv')):
        df = charger_logs(date, NORMAL, f.stem.replace('_log', ''))
        if df.empty:
            continue
        df['template'] = df['Log'].apply(extraire_template)
        textes_normaux_ob.append(' '.join(df['template'].tolist()))

tfidf_ob = TfidfVectorizer(max_features=500)
tfidf_ob.fit(textes_normaux_ob)
vecteur_ref_ob = np.asarray(tfidf_ob.transform(textes_normaux_ob).mean(axis=0))

with open(MODELS_DIR / 'tfidf_ob.pkl', 'wb') as f:
    pickle.dump({'vectorizer': tfidf_ob, 'vecteur_ref': vecteur_ref_ob}, f)
print(f"  ✓ TF-IDF sauvegardé ({len(tfidf_ob.vocabulary_)} termes)")

# ─── IF par service Online Boutique ───
print("\n=== Isolation Forest par service Online Boutique ===")
modeles_if_traces_ob = {}
scalers_if_traces_ob = {}
for date in DATES_OB:
    trace_dir = NORMAL / date / 'trace'
    if not trace_dir.exists():
        continue
    for f in sorted(trace_dir.glob('*.csv')):
        df = charger_traces(date, NORMAL, f.stem.replace('_trace', ''))
        if df.empty:
            continue
        for service in df['service'].unique():
            df_svc = df[df['service'] == service][FEATURES_SPAN].dropna()
            if len(df_svc) < 5:
                continue
            if service not in modeles_if_traces_ob:
                scaler = StandardScaler()
                X_svc = scaler.fit_transform(df_svc)
                model = IsolationForest(
                    n_estimators=100, contamination=0.10, random_state=42
                )
                model.fit(X_svc)
                modeles_if_traces_ob[service] = model
                scalers_if_traces_ob[service] = scaler

with open(MODELS_DIR / 'if_traces_ob.pkl', 'wb') as f:
    pickle.dump({'modeles': modeles_if_traces_ob, 'scalers': scalers_if_traces_ob}, f)
print(f"  ✓ {len(modeles_if_traces_ob)} modèles IF sauvegardés")

# ─── Récapitulatif ───
print("\n" + "="*50)
print("RÉCAPITULATIF")
print("="*50)
print(f"  Train Ticket   : LOF ({len(modeles_lof_tt)}) + TF-IDF + IF ({len(modeles_if_traces_tt)})")
print(f"  Online Boutique: LOF ({len(modeles_lof_ob)}) + TF-IDF + IF ({len(modeles_if_traces_ob)})")
print(f"\n  6 fichiers .pkl dans models/")

=== LOF Online Boutique ===
  Données : 19,300 lignes, 10 services
  ✓ 10 modèles LOF sauvegardés

=== TF-IDF Online Boutique ===
  ✓ TF-IDF sauvegardé (15 termes)

=== Isolation Forest par service Online Boutique ===
  ✓ 10 modèles IF sauvegardés

RÉCAPITULATIF
  Train Ticket   : LOF (46) + TF-IDF + IF (28)
  Online Boutique: LOF (10) + TF-IDF + IF (10)

  6 fichiers .pkl dans models/
